# 🚢 EDA — Predição de Lead Time Portuário
**TCC | Análise Exploratória de Dados**

Este notebook realiza uma EDA profunda sobre os dados de estadia de embarcações em portos brasileiros, investigando as principais variáveis que influenciam o lead time e o capital de giro.

---
**Seções:**
1. Setup & Carga de Dados
2. Visão Geral do Dataset
3. Análise Univariada (Targets)
4. Análise por Região Geográfica + Mapas
5. Sazonalidade e Clima
6. Análise por Tipo de Carga / Mercadoria
7. Análise por Tipo de Embarcação
8. Matriz de Correlação & Feature Importance Exploratória
9. Outliers & Anomalias
10. Sugestões de Feature Engineering
---

## 0. Instalação de Dependências

In [ ]:
# Instale caso não tenha
#!pip install folium geopandas plotly kaleido missingno scikit-learn statsmodels
#!pip install branca contextily

   ---------------------------------------- 0.0/30.1 MB ? eta -:--:--
   ------------------ --------------------- 13.9/30.1 MB 70.7 MB/s eta 0:00:01
   ------------------------------------- -- 28.6/30.1 MB 71.3 MB/s eta 0:00:01
   ---------------------------------------- 30.1/30.1 MB 63.7 MB/s  0:00:00

   -------------------- ------------------- 4/8 [geopy]
   ------------------------------ --------- 6/8 [rasterio]
   ------------------------------ --------- 6/8 [rasterio]
   ------------------------------ --------- 6/8 [rasterio]
   ------------------------------ --------- 6/8 [rasterio]
   ---------------------------------------- 8/8 [contextily]



## 1. Setup & Carga de Dados

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.cm as cm
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
import folium
from folium.plugins import HeatMap, MarkerCluster
import missingno as msno
from scipy import stats
from sklearn.preprocessing import LabelEncoder
import json

warnings.filterwarnings('ignore')

# ── Estilo Global ──────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 130,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 14,
    'axes.labelsize': 12,
})

PALETTE = ['#0077B6', '#00B4D8', '#90E0EF', '#023E8A', '#ADE8F4',
           '#FF6B6B', '#FFA552', '#FFD166', '#06D6A0', '#118AB2']
sns.set_palette(PALETTE)

print('✅ Bibliotecas carregadas com sucesso!')

In [ ]:
# ── Carga dos Dados Processados ────────────────────────────────────────────
# Ajuste o caminho conforme necessário
DATA_PATH = '../data/processed/estadia_embarcacao_processed.parquet'

try:
    df = pd.read_parquet(DATA_PATH)
    print(f'✅ Dados carregados: {df.shape[0]:,} linhas × {df.shape[1]} colunas')
except FileNotFoundError:
    # Fallback: tenta CSV
    DATA_PATH = '../data/processed/estadia_embarcacao_processed.csv'
    df = pd.read_csv(DATA_PATH, low_memory=False)
    print(f'✅ Dados carregados (CSV): {df.shape[0]:,} linhas × {df.shape[1]} colunas')

df.head(3)

## 2. Visão Geral do Dataset

In [ ]:
# ── Resumo Estatístico ─────────────────────────────────────────────────────
print('=== TIPOS DE DADOS ===')
print(df.dtypes.value_counts())
print()
print('=== COLUNAS ===')
for col in df.columns:
    nulls = df[col].isna().sum()
    pct = nulls / len(df) * 100
    print(f'  {col:<45} | nulos: {nulls:>6} ({pct:.1f}%)')

In [ ]:
# ── Mapa de Missings ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

msno.matrix(df, ax=axes[0], color=(0, 0.47, 0.71), fontsize=9)
axes[0].set_title('Mapa de Valores Ausentes', fontsize=14, fontweight='bold')

missing_pct = df.isna().mean().sort_values(ascending=False) * 100
missing_pct = missing_pct[missing_pct > 0]
if len(missing_pct) > 0:
    axes[1].barh(missing_pct.index, missing_pct.values,
                 color=['#FF6B6B' if v > 30 else '#FFA552' if v > 10 else '#FFD166'
                        for v in missing_pct.values])
    axes[1].set_xlabel('% de Ausentes')
    axes[1].set_title('% de Missing por Coluna', fontsize=14, fontweight='bold')
    axes[1].axvline(30, ls='--', color='#FF6B6B', alpha=0.7, label='>30% crítico')
    axes[1].legend()
else:
    axes[1].text(0.5, 0.5, 'Nenhum missing!', ha='center', va='center',
                 fontsize=16, transform=axes[1].transAxes, color='#06D6A0')
    axes[1].axis('off')

plt.tight_layout()
plt.savefig('../reports/figures/01_missing_data.png', bbox_inches='tight')
plt.show()

## 3. Análise Univariada — Variáveis Target

In [ ]:
# ── ATENÇÃO: Ajuste os nomes das colunas target conforme seu dataset ───────
# Exemplos comuns para dados de estadia portuária:
# TARGET_COLS = ['lead_time_horas', 'estadia_total_horas', 'tempo_operacao_horas']
#
# Detecção automática de candidatos a target (colunas numéricas com 'tempo', 'estadia', 'lead')
import re
TARGET_CANDIDATES = [c for c in df.columns
                     if re.search(r'tempo|estadia|lead|duracao|hora|dia', c, re.I)
                     and df[c].dtype in ['float64', 'int64']]

print('Candidatos a target identificados automaticamente:')
for t in TARGET_CANDIDATES:
    print(f'  {t}: min={df[t].min():.1f}, max={df[t].max():.1f}, median={df[t].median():.1f}')

In [ ]:
# Defina manualmente se necessário:
# TARGET_COLS = ['nome_da_sua_coluna_target']
TARGET_COLS = TARGET_CANDIDATES[:3]  # pega até 3 candidatos

for target in TARGET_COLS:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f'Distribuição: {target}', fontsize=16, fontweight='bold', y=1.02)

    data = df[target].dropna()

    # Histograma + KDE
    axes[0].hist(data, bins=60, color='#0077B6', alpha=0.8, edgecolor='white', linewidth=0.3)
    ax2 = axes[0].twinx()
    data.plot.kde(ax=ax2, color='#FF6B6B', linewidth=2.5)
    axes[0].set_title('Histograma + KDE')
    axes[0].set_xlabel(target)
    ax2.set_ylabel('Densidade')

    # Log-transformed
    log_data = np.log1p(data)
    axes[1].hist(log_data, bins=60, color='#00B4D8', alpha=0.8, edgecolor='white', linewidth=0.3)
    axes[1].set_title('Distribuição Log(1+x)')
    axes[1].set_xlabel(f'log(1 + {target})')

    # Boxplot
    bp = axes[2].boxplot(data, vert=True, patch_artist=True, widths=0.6,
                         boxprops=dict(facecolor='#90E0EF', color='#023E8A'),
                         medianprops=dict(color='#FF6B6B', linewidth=2.5),
                         whiskerprops=dict(color='#023E8A'),
                         capprops=dict(color='#023E8A'),
                         flierprops=dict(marker='o', markersize=2,
                                         markerfacecolor='#FFA552', alpha=0.3))
    axes[2].set_title('Boxplot')
    axes[2].set_ylabel(target)

    # Stats box
    stats_text = (f'n = {len(data):,}\n'
                  f'Média = {data.mean():.1f}\n'
                  f'Mediana = {data.median():.1f}\n'
                  f'Std = {data.std():.1f}\n'
                  f'Skew = {data.skew():.2f}\n'
                  f'Kurt = {data.kurt():.2f}')
    axes[2].text(1.35, 0.5, stats_text, transform=axes[2].transAxes,
                 fontsize=10, va='center', bbox=dict(boxstyle='round', facecolor='#ADE8F4', alpha=0.5))

    plt.tight_layout()
    plt.savefig(f'../reports/figures/02_dist_{target}.png', bbox_inches='tight')
    plt.show()

In [ ]:
# ── Q-Q Plot para verificar normalidade ───────────────────────────────────
if TARGET_COLS:
    target = TARGET_COLS[0]
    data = df[target].dropna()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'Q-Q Plot — {target}', fontsize=14, fontweight='bold')

    stats.probplot(data, dist='norm', plot=axes[0])
    axes[0].set_title('Original')
    axes[0].get_lines()[0].set(markerfacecolor='#0077B6', markersize=2, alpha=0.4)
    axes[0].get_lines()[1].set(color='#FF6B6B', linewidth=2)

    stats.probplot(np.log1p(data), dist='norm', plot=axes[1])
    axes[1].set_title('Log-transformado')
    axes[1].get_lines()[0].set(markerfacecolor='#00B4D8', markersize=2, alpha=0.4)
    axes[1].get_lines()[1].set(color='#FF6B6B', linewidth=2)

    plt.tight_layout()
    plt.savefig(f'../reports/figures/03_qqplot_{target}.png', bbox_inches='tight')
    plt.show()

## 4. Análise por Região Geográfica + Mapas

> Coordenadas aproximadas dos principais portos brasileiros estão embutidas abaixo como referência.

In [ ]:
# ── Dicionário de Portos Brasileiros ──────────────────────────────────────
# Ajuste os nomes conforme sua coluna de porto/instalação
PORTOS_GEO = {
    # Porto do Norte
    'Santarém':        {'lat': -2.44,  'lon': -54.71, 'regiao': 'Norte',    'estado': 'PA'},
    'Belém':           {'lat': -1.45,  'lon': -48.50, 'regiao': 'Norte',    'estado': 'PA'},
    'Manaus':          {'lat': -3.10,  'lon': -60.02, 'regiao': 'Norte',    'estado': 'AM'},
    'Macapá':          {'lat':  0.03,  'lon': -51.07, 'regiao': 'Norte',    'estado': 'AP'},
    'Itacoatiara':     {'lat': -3.14,  'lon': -58.44, 'regiao': 'Norte',    'estado': 'AM'},
    # Nordeste
    'Fortaleza':       {'lat': -3.72,  'lon': -38.53, 'regiao': 'Nordeste', 'estado': 'CE'},
    'Recife':          {'lat': -8.05,  'lon': -34.88, 'regiao': 'Nordeste', 'estado': 'PE'},
    'Salvador':        {'lat': -12.97, 'lon': -38.50, 'regiao': 'Nordeste', 'estado': 'BA'},
    'São Luís':        {'lat': -2.54,  'lon': -44.30, 'regiao': 'Nordeste', 'estado': 'MA'},
    'Natal':           {'lat': -5.79,  'lon': -35.21, 'regiao': 'Nordeste', 'estado': 'RN'},
    'Suape':           {'lat': -8.39,  'lon': -34.97, 'regiao': 'Nordeste', 'estado': 'PE'},
    'Maceió':          {'lat': -9.66,  'lon': -35.74, 'regiao': 'Nordeste', 'estado': 'AL'},
    'Aratu':           {'lat': -12.77, 'lon': -38.50, 'regiao': 'Nordeste', 'estado': 'BA'},
    # Sudeste
    'Santos':          {'lat': -23.95, 'lon': -46.33, 'regiao': 'Sudeste',  'estado': 'SP'},
    'Rio de Janeiro':  {'lat': -22.89, 'lon': -43.19, 'regiao': 'Sudeste',  'estado': 'RJ'},
    'Vitória':         {'lat': -20.32, 'lon': -40.33, 'regiao': 'Sudeste',  'estado': 'ES'},
    'Angra dos Reis':  {'lat': -23.00, 'lon': -44.32, 'regiao': 'Sudeste',  'estado': 'RJ'},
    'Sepetiba':        {'lat': -22.95, 'lon': -43.72, 'regiao': 'Sudeste',  'estado': 'RJ'},
    'Itaguaí':         {'lat': -22.87, 'lon': -43.79, 'regiao': 'Sudeste',  'estado': 'RJ'},
    # Sul
    'Paranaguá':       {'lat': -25.52, 'lon': -48.51, 'regiao': 'Sul',      'estado': 'PR'},
    'São Francisco do Sul': {'lat': -26.24, 'lon': -48.64, 'regiao': 'Sul', 'estado': 'SC'},
    'Itajaí':          {'lat': -26.91, 'lon': -48.67, 'regiao': 'Sul',      'estado': 'SC'},
    'Imbituba':        {'lat': -28.23, 'lon': -48.66, 'regiao': 'Sul',      'estado': 'SC'},
    'Porto Alegre':    {'lat': -30.03, 'lon': -51.23, 'regiao': 'Sul',      'estado': 'RS'},
    'Rio Grande':      {'lat': -32.03, 'lon': -52.10, 'regiao': 'Sul',      'estado': 'RS'},
    # Centro-Oeste
    'Porto Velho':     {'lat': -8.76,  'lon': -63.90, 'regiao': 'Norte',    'estado': 'RO'},
}

portos_df = pd.DataFrame(PORTOS_GEO).T.reset_index().rename(columns={'index': 'porto'})
portos_df['lat'] = portos_df['lat'].astype(float)
portos_df['lon'] = portos_df['lon'].astype(float)

print(f'Portos no dicionário de referência: {len(portos_df)}')
portos_df.head()

In [ ]:
# ── Identifica coluna de porto/instalação no dataset ──────────────────────
PORTO_COL = None
for candidate in ['porto', 'instalacao', 'cd_porto', 'nm_porto', 'porto_atracacao',
                  'instalacao_portuaria', 'nome_porto', 'terminal']:
    if candidate in df.columns:
        PORTO_COL = candidate
        break

if PORTO_COL:
    print(f'Coluna de porto identificada: {PORTO_COL}')
    print(df[PORTO_COL].value_counts().head(20))
else:
    print('⚠️  Coluna de porto não encontrada automaticamente. Defina manualmente:')
    print('   PORTO_COL = "nome_da_coluna"')

In [ ]:
# ── Agrega métricas por porto ──────────────────────────────────────────────
if PORTO_COL and TARGET_COLS:
    target = TARGET_COLS[0]
    porto_stats = (df.groupby(PORTO_COL)[target]
                     .agg(['median', 'mean', 'std', 'count'])
                     .reset_index()
                     .rename(columns={'median': 'lead_time_mediano',
                                      'mean': 'lead_time_medio',
                                      'std': 'lead_time_std',
                                      'count': 'n_escalas'}))

    # Faz o merge com coordenadas (match fuzzy simples por similaridade de nome)
    porto_stats['porto_match'] = porto_stats[PORTO_COL].str.title()
    porto_geo = porto_stats.merge(
        portos_df.rename(columns={'porto': 'porto_match'}),
        on='porto_match', how='left'
    )

    matched = porto_geo['lat'].notna().sum()
    print(f'Portos com coordenadas encontradas: {matched}/{len(porto_geo)}')
    porto_geo.head()
else:
    print('Defina PORTO_COL e TARGET_COLS para continuar esta análise.')

In [ ]:
# ── Mapa Interativo Folium — Heatmap de Lead Time por Porto ──────────────
if PORTO_COL and TARGET_COLS and 'porto_geo' in dir():
    porto_geo_valid = porto_geo.dropna(subset=['lat', 'lon'])

    m = folium.Map(location=[-15, -50], zoom_start=4,
                   tiles='CartoDB positron')

    # HeatMap de intensidade de escalas
    heat_data = [[row['lat'], row['lon'], row['n_escalas']]
                 for _, row in porto_geo_valid.iterrows()]
    HeatMap(heat_data, min_opacity=0.3, radius=30, blur=25,
            gradient={'0.2': '#ADE8F4', '0.5': '#0077B6', '1.0': '#FF6B6B'}).add_to(m)

    # Círculos proporcionais ao lead time mediano
    for _, row in porto_geo_valid.iterrows():
        lt = row['lead_time_mediano']
        color = '#FF6B6B' if lt > porto_geo_valid['lead_time_mediano'].quantile(0.75) \
                else '#FFA552' if lt > porto_geo_valid['lead_time_mediano'].median() \
                else '#06D6A0'

        folium.CircleMarker(
            location=[row['lat'], row['lon']],
            radius=max(5, min(30, row['n_escalas'] / 200)),
            color=color, fill=True, fill_opacity=0.7,
            popup=folium.Popup(
                f"<b>{row[PORTO_COL]}</b><br>"
                f"Lead Time Mediano: <b>{lt:.1f}h</b><br>"
                f"Escalas: {row['n_escalas']:,}",
                max_width=200
            ),
            tooltip=row[PORTO_COL]
        ).add_to(m)

    # Legenda
    legend_html = '''
    <div style="position:fixed;bottom:30px;left:30px;z-index:1000;
                background:white;padding:12px;border-radius:8px;
                box-shadow:2px 2px 6px rgba(0,0,0,0.3);font-size:13px">
      <b>🚢 Lead Time por Porto</b><br>
      <span style="color:#FF6B6B">●</span> Alto (>Q75)<br>
      <span style="color:#FFA552">●</span> Médio<br>
      <span style="color:#06D6A0">●</span> Baixo<br>
      <i>Tamanho = volume de escalas</i>
    </div>'''
    m.get_root().html.add_child(folium.Element(legend_html))

    m.save('../reports/figures/mapa_lead_time_portos.html')
    print('✅ Mapa salvo em ../reports/figures/mapa_lead_time_portos.html')
    m
else:
    print('Pule este bloco se as colunas não foram identificadas.')

In [ ]:
# ── Gráfico de Barras: Lead Time Mediano por Porto (Top 20) ───────────────
if PORTO_COL and TARGET_COLS:
    target = TARGET_COLS[0]
    top_portos = (df.groupby(PORTO_COL)[target]
                    .median()
                    .sort_values(ascending=False)
                    .head(20))

    fig, ax = plt.subplots(figsize=(14, 7))
    colors = [PALETTE[0] if v > top_portos.median() * 1.5
              else PALETTE[3] if v > top_portos.median()
              else PALETTE[2]
              for v in top_portos.values]

    bars = ax.barh(top_portos.index, top_portos.values, color=colors,
                   edgecolor='white', linewidth=0.5)

    for bar, val in zip(bars, top_portos.values):
        ax.text(val + top_portos.max() * 0.01, bar.get_y() + bar.get_height() / 2,
                f'{val:.1f}h', va='center', fontsize=9, color='#333')

    ax.axvline(top_portos.median(), ls='--', color='#FF6B6B', lw=1.5,
               label=f'Mediana Geral ({top_portos.median():.1f}h)')
    ax.set_xlabel(f'Lead Time Mediano ({target})', fontsize=12)
    ax.set_title('🏆 Top 20 Portos por Lead Time Mediano', fontsize=15, fontweight='bold')
    ax.legend()
    ax.invert_yaxis()

    plt.tight_layout()
    plt.savefig('../reports/figures/04_leadtime_por_porto.png', bbox_inches='tight')
    plt.show()

In [ ]:
# ── Box Plot: Distribuição de Lead Time por Região ─────────────────────────
# Primeiro, tenta identificar coluna de região;
# se não existir, faz o cruzamento com o dicionário de portos

REGIAO_COL = None
for c in ['regiao', 'nm_regiao', 'cd_regiao', 'regiao_geografica']:
    if c in df.columns:
        REGIAO_COL = c
        break

# Se não existir, tenta criar via merge com dicionário
if REGIAO_COL is None and PORTO_COL and 'porto_geo' in dir():
    df_plot = df.merge(
        porto_geo[[PORTO_COL, 'regiao']].drop_duplicates(),
        on=PORTO_COL, how='left'
    )
    REGIAO_COL = 'regiao'
    print('Região atribuída via dicionário de portos.')
else:
    df_plot = df.copy()

if REGIAO_COL and REGIAO_COL in df_plot.columns and TARGET_COLS:
    target = TARGET_COLS[0]
    order = (df_plot.groupby(REGIAO_COL)[target]
                    .median()
                    .sort_values(ascending=False)
                    .index.tolist())

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(f'Lead Time por Região — {target}', fontsize=15, fontweight='bold')

    # Violin
    sns.violinplot(data=df_plot, x=REGIAO_COL, y=target, order=order,
                   palette=PALETTE[:len(order)], inner='quartile', ax=axes[0])
    axes[0].set_title('Violin Plot')
    axes[0].set_xlabel('Região')
    axes[0].set_ylabel('Lead Time')

    # Boxplot
    sns.boxplot(data=df_plot, x=REGIAO_COL, y=target, order=order,
                palette=PALETTE[:len(order)], ax=axes[1], fliersize=2)
    axes[1].set_title('Box Plot')
    axes[1].set_xlabel('Região')
    axes[1].set_ylabel('')

    plt.tight_layout()
    plt.savefig('../reports/figures/05_leadtime_por_regiao.png', bbox_inches='tight')
    plt.show()

In [ ]:
# ── Mapa Choropleth Interativo (Plotly) por Estado ─────────────────────────
if PORTO_COL and TARGET_COLS and 'porto_geo' in dir():
    target = TARGET_COLS[0]
    # Agrega por estado
    estado_stats = (porto_geo.dropna(subset=['estado'])
                             .groupby('estado')
                             .agg(lead_time_mediano=('lead_time_mediano', 'median'),
                                  n_escalas=('n_escalas', 'sum'))
                             .reset_index())

    fig = px.choropleth(
        estado_stats,
        geojson='https://raw.githubusercontent.com/codeforamerica/click_that_hood/master/public/data/brazil-states.geojson',
        locations='estado',
        featureidkey='properties.sigla',
        color='lead_time_mediano',
        hover_data=['n_escalas'],
        color_continuous_scale=[
            [0.0, '#06D6A0'], [0.5, '#FFD166'], [1.0, '#FF6B6B']
        ],
        title='🗺️ Lead Time Mediano por Estado Brasileiro',
        labels={'lead_time_mediano': 'Lead Time (h)', 'n_escalas': 'Nº Escalas'}
    )
    fig.update_geos(fitbounds='locations', visible=False)
    fig.update_layout(
        height=600,
        font_family='Arial',
        title_font_size=18,
        coloraxis_colorbar_title='Lead Time (h)'
    )
    fig.write_html('../reports/figures/mapa_choropleth_estados.html')
    fig.show()
    print('✅ Mapa choropleth salvo!')

## 5. Sazonalidade e Clima

In [ ]:
# ── Identifica colunas de data ─────────────────────────────────────────────
DATE_COL = None
for c in ['data_atracacao', 'dt_atracacao', 'data_inicio', 'dt_inicio',
          'data_chegada', 'dt_chegada', 'data', 'dt_operacao']:
    if c in df.columns:
        DATE_COL = c
        break

if DATE_COL:
    df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors='coerce')
    df['ano']  = df[DATE_COL].dt.year
    df['mes']  = df[DATE_COL].dt.month
    df['dow']  = df[DATE_COL].dt.dayofweek   # 0=seg
    df['hora_dia'] = df[DATE_COL].dt.hour
    df['trimestre'] = df[DATE_COL].dt.quarter

    # Mapeamento de estação climática (hemisfério Sul)
    df['estacao'] = df['mes'].map({
        12: 'Verão', 1: 'Verão', 2: 'Verão',
        3: 'Outono', 4: 'Outono', 5: 'Outono',
        6: 'Inverno', 7: 'Inverno', 8: 'Inverno',
        9: 'Primavera', 10: 'Primavera', 11: 'Primavera'
    })

    print(f'✅ Coluna de data: {DATE_COL}')
    print(f'   Período: {df[DATE_COL].min().date()} → {df[DATE_COL].max().date()}')
else:
    print('⚠️  Coluna de data não encontrada. Defina manualmente: DATE_COL = "nome_da_coluna"')

In [ ]:
# ── Sazonalidade: Lead Time por Mês e Dia da Semana ──────────────────────
if DATE_COL and TARGET_COLS:
    target = TARGET_COLS[0]

    fig, axes = plt.subplots(2, 2, figsize=(18, 12))
    fig.suptitle('⏰ Sazonalidade do Lead Time', fontsize=18, fontweight='bold')

    # ① Por mês
    mes_stats = df.groupby('mes')[target].agg(['median', 'mean', 'std'])
    meses = ['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun',
             'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez']
    axes[0, 0].bar(range(1, 13), mes_stats['median'],
                   color=[PALETTE[5] if v > mes_stats['median'].median() else PALETTE[2]
                          for v in mes_stats['median']],
                   edgecolor='white')
    axes[0, 0].errorbar(range(1, 13), mes_stats['median'],
                        yerr=mes_stats['std'] / 2, fmt='none', color='#333', capsize=4)
    axes[0, 0].set_xticks(range(1, 13))
    axes[0, 0].set_xticklabels(meses)
    axes[0, 0].set_title('① Lead Time Mediano por Mês', fontweight='bold')
    axes[0, 0].set_ylabel('Lead Time Mediano')

    # ② Por dia da semana
    dias = ['Seg', 'Ter', 'Qua', 'Qui', 'Sex', 'Sáb', 'Dom']
    dow_stats = df.groupby('dow')[target].median()
    axes[0, 1].bar(dias, dow_stats.reindex(range(7)).values,
                   color=[PALETTE[5] if i >= 5 else PALETTE[0] for i in range(7)],
                   edgecolor='white')
    axes[0, 1].set_title('② Lead Time Mediano por Dia da Semana', fontweight='bold')
    axes[0, 1].set_ylabel('Lead Time Mediano')

    # ③ Por estação
    estacao_order = ['Verão', 'Outono', 'Inverno', 'Primavera']
    estacao_colors = ['#FF6B6B', '#FFA552', '#0077B6', '#06D6A0']
    est_stats = df.groupby('estacao')[target].median().reindex(estacao_order)
    axes[1, 0].bar(estacao_order, est_stats.values, color=estacao_colors, edgecolor='white')
    axes[1, 0].set_title('③ Lead Time Mediano por Estação', fontweight='bold')
    axes[1, 0].set_ylabel('Lead Time Mediano')

    # ④ Heatmap Mês × Dia da Semana
    pivot = df.groupby(['mes', 'dow'])[target].median().unstack()
    pivot.index = meses
    pivot.columns = dias[:pivot.shape[1]]
    sns.heatmap(pivot, ax=axes[1, 1],
                cmap='RdYlGn_r', annot=True, fmt='.0f',
                linewidths=0.5, cbar_kws={'label': 'Lead Time Mediano'})
    axes[1, 1].set_title('④ Heatmap: Mês × Dia da Semana', fontweight='bold')

    plt.tight_layout()
    plt.savefig('../reports/figures/06_sazonalidade.png', bbox_inches='tight')
    plt.show()

In [ ]:
# ── Série Temporal Agregada (mensal) ──────────────────────────────────────
if DATE_COL and TARGET_COLS:
    target = TARGET_COLS[0]
    ts = df.set_index(DATE_COL)[target].resample('ME').agg(['median', 'count'])
    ts.columns = ['lead_time_mediano', 'n_escalas']

    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        subplot_titles=['Lead Time Mediano Mensal',
                                        'Volume de Escalas Mensais'],
                        vertical_spacing=0.08)

    fig.add_trace(go.Scatter(
        x=ts.index, y=ts['lead_time_mediano'],
        mode='lines+markers', name='Lead Time Mediano',
        line=dict(color='#0077B6', width=2.5),
        marker=dict(size=4)
    ), row=1, col=1)

    # Rolling average
    rolling = ts['lead_time_mediano'].rolling(3, center=True).mean()
    fig.add_trace(go.Scatter(
        x=ts.index, y=rolling, mode='lines', name='Média Móvel 3m',
        line=dict(color='#FF6B6B', width=2, dash='dash')
    ), row=1, col=1)

    fig.add_trace(go.Bar(
        x=ts.index, y=ts['n_escalas'],
        name='Nº Escalas', marker_color='#90E0EF',
        opacity=0.8
    ), row=2, col=1)

    fig.update_layout(
        height=600, title='📈 Evolução Temporal do Lead Time Portuário',
        title_font_size=18, hovermode='x unified',
        legend=dict(orientation='h', y=1.02)
    )
    fig.write_html('../reports/figures/serie_temporal_leadtime.html')
    fig.show()

## 6. Análise por Tipo de Carga / Mercadoria

In [ ]:
# ── Identifica coluna de tipo de carga ────────────────────────────────────
CARGA_COL = None
for c in ['tipo_carga', 'cd_tipo_carga', 'nm_tipo_carga', 'mercadoria',
          'tipo_mercadoria', 'natureza_carga', 'carga']:
    if c in df.columns:
        CARGA_COL = c
        break

if CARGA_COL and TARGET_COLS:
    target = TARGET_COLS[0]
    top_cargas = df[CARGA_COL].value_counts().head(15).index
    df_carga = df[df[CARGA_COL].isin(top_cargas)]

    carga_order = (df_carga.groupby(CARGA_COL)[target]
                           .median()
                           .sort_values(ascending=False)
                           .index.tolist())

    fig, axes = plt.subplots(1, 2, figsize=(20, 8))
    fig.suptitle(f'🏋️ Lead Time por Tipo de Carga', fontsize=16, fontweight='bold')

    # Boxplot
    sns.boxplot(data=df_carga, x=target, y=CARGA_COL, order=carga_order,
                palette='Blues_r', fliersize=1.5, ax=axes[0])
    axes[0].set_title('Distribuição por Tipo de Carga')
    axes[0].set_xlabel('Lead Time')

    # Dot plot com CI
    carga_stats = df_carga.groupby(CARGA_COL)[target].agg(['median', 'count',
        lambda x: x.quantile(0.25), lambda x: x.quantile(0.75)])
    carga_stats.columns = ['median', 'count', 'q25', 'q75']
    carga_stats = carga_stats.reindex(carga_order)

    y_pos = range(len(carga_stats))
    axes[1].barh(y_pos, carga_stats['median'],
                 xerr=[carga_stats['median'] - carga_stats['q25'],
                       carga_stats['q75'] - carga_stats['median']],
                 color=PALETTE[0], alpha=0.8, capsize=5, error_kw={'linewidth': 2})
    axes[1].set_yticks(y_pos)
    axes[1].set_yticklabels(carga_stats.index)
    axes[1].set_title('Mediana + IQR por Tipo de Carga')
    axes[1].set_xlabel('Lead Time Mediano (+ IQR)')
    axes[1].invert_yaxis()

    plt.tight_layout()
    plt.savefig('../reports/figures/07_leadtime_tipo_carga.png', bbox_inches='tight')
    plt.show()
else:
    print(f'⚠️  Coluna de tipo de carga não encontrada. Identificadas no dataset: {list(df.columns)}')

## 7. Análise por Tipo de Embarcação

In [ ]:
# ── Identifica coluna de tipo de embarcação ───────────────────────────────
EMBARCACAO_COL = None
for c in ['tipo_embarcacao', 'cd_tipo_embarcacao', 'nm_tipo_embarcacao',
          'tipo_navio', 'tipo_vessel', 'embarcacao']:
    if c in df.columns:
        EMBARCACAO_COL = c
        break

if EMBARCACAO_COL and TARGET_COLS:
    target = TARGET_COLS[0]
    emb_stats = (df.groupby(EMBARCACAO_COL)[target]
                   .agg(['count', 'median', 'mean', 'std'])
                   .reset_index()
                   .sort_values('median', ascending=False)
                   .head(15))

    # Bubble Chart interativo
    fig = px.scatter(
        emb_stats,
        x='median', y='std',
        size='count', color='median',
        hover_name=EMBARCACAO_COL,
        color_continuous_scale='RdYlGn_r',
        size_max=60,
        labels={'median': 'Lead Time Mediano (h)',
                'std': 'Desvio Padrão (h)',
                'count': 'Nº Escalas'},
        title='🚢 Tipo de Embarcação: Lead Time Mediano × Variabilidade'
    )
    fig.update_layout(height=500, title_font_size=16)
    fig.write_html('../reports/figures/bubble_tipo_embarcacao.html')
    fig.show()
else:
    print('⚠️  Coluna de tipo de embarcação não identificada automaticamente.')

In [ ]:
# ── Análise: Tonelagem / DWT vs Lead Time ─────────────────────────────────
DWT_COL = None
for c in ['dwt', 'porte_bruto', 'tonelagem', 'gt', 'gross_tonnage',
          'capacidade', 'teu', 'capacidade_teu']:
    if c in df.columns:
        DWT_COL = c
        break

if DWT_COL and TARGET_COLS:
    target = TARGET_COLS[0]
    sample = df[[DWT_COL, target]].dropna().sample(min(5000, len(df)))

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(f'Tonelagem/Porte ({DWT_COL}) × Lead Time', fontsize=15, fontweight='bold')

    # Scatter com regressão
    axes[0].scatter(sample[DWT_COL], sample[target],
                    alpha=0.3, s=10, color=PALETTE[0])
    m, b = np.polyfit(np.log1p(sample[DWT_COL]), sample[target], 1)
    x_range = np.linspace(sample[DWT_COL].min(), sample[DWT_COL].max(), 200)
    axes[0].plot(x_range, m * np.log1p(x_range) + b,
                 color='#FF6B6B', linewidth=2.5, label='Ajuste log-linear')
    axes[0].set_xlabel(DWT_COL)
    axes[0].set_ylabel(target)
    axes[0].legend()
    axes[0].set_title('Scatter com Ajuste Log-linear')

    # Hexbin
    hb = axes[1].hexbin(np.log1p(sample[DWT_COL]), sample[target],
                        gridsize=35, cmap='YlOrRd', mincnt=1)
    plt.colorbar(hb, ax=axes[1], label='Frequência')
    axes[1].set_xlabel(f'log(1 + {DWT_COL})')
    axes[1].set_ylabel(target)
    axes[1].set_title('Hexbin — Densidade')

    plt.tight_layout()
    plt.savefig('../reports/figures/08_tonelagem_vs_leadtime.png', bbox_inches='tight')
    plt.show()

## 8. Matriz de Correlação & Feature Importance Exploratória

In [ ]:
# ── Matriz de Correlação (numéricas) ──────────────────────────────────────
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr_matrix = df[num_cols].corr()

# Filtra apenas colunas com correlação relevante com os targets
if TARGET_COLS:
    target = TARGET_COLS[0]
    relevant_corrs = corr_matrix[target].abs().sort_values(ascending=False)
    top_feats = relevant_corrs.head(20).index.tolist()
    corr_subset = corr_matrix.loc[top_feats, top_feats]
else:
    corr_subset = corr_matrix

fig, ax = plt.subplots(figsize=(14, 11))

mask = np.triu(np.ones_like(corr_subset, dtype=bool))
sns.heatmap(
    corr_subset, mask=mask, annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    linewidths=0.5, ax=ax,
    annot_kws={'size': 8},
    cbar_kws={'label': 'Coeficiente de Correlação'}
)
ax.set_title('🔗 Matriz de Correlação (Top Features × Target)',
             fontsize=15, fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/figures/09_correlation_matrix.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Feature Importance com RandomForest (exploratória) ────────────────────
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder

if TARGET_COLS:
    target = TARGET_COLS[0]

    # Prepara um dataset simplificado para RF exploratório
    df_rf = df.copy()

    # Encode categoricals
    cat_cols = df_rf.select_dtypes(include='object').columns.tolist()
    le = LabelEncoder()
    for col in cat_cols:
        df_rf[col] = df_rf[col].fillna('__missing__')
        df_rf[col] = le.fit_transform(df_rf[col].astype(str))

    # Remove target e colunas de data
    drop_cols = [c for c in df_rf.columns if 'data' in c.lower() or 'dt_' in c.lower()]
    feat_cols = [c for c in df_rf.columns if c != target and c not in drop_cols
                 and c in num_cols + cat_cols]

    df_rf_clean = df_rf[feat_cols + [target]].dropna().sample(min(20_000, len(df_rf)))
    X = df_rf_clean[feat_cols]
    y = df_rf_clean[target]

    rf = RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1)
    rf.fit(X, y)

    importances = pd.Series(rf.feature_importances_, index=feat_cols).sort_values(ascending=False)

    fig, ax = plt.subplots(figsize=(12, 8))
    top_imp = importances.head(20)
    colors_imp = [PALETTE[0] if v > top_imp.median() else PALETTE[2]
                  for v in top_imp.values]
    top_imp.sort_values().plot.barh(ax=ax, color=list(reversed(colors_imp)))
    ax.set_title('🌲 Feature Importance — Random Forest Exploratório', fontsize=15, fontweight='bold')
    ax.set_xlabel('Importância Relativa')
    ax.axvline(top_imp.median(), ls='--', color='#FF6B6B', lw=1.5, label='Mediana')
    ax.legend()

    plt.tight_layout()
    plt.savefig('../reports/figures/10_feature_importance.png', bbox_inches='tight')
    plt.show()

    print('\n📊 Top 10 Features mais importantes:')
    for feat, imp in importances.head(10).items():
        print(f'   {feat:<40} {imp:.4f}')

## 9. Outliers & Anomalias

In [ ]:
# ── Detecção de Outliers — IQR e Z-Score ──────────────────────────────────
if TARGET_COLS:
    target = TARGET_COLS[0]
    data = df[target].dropna()

    Q1, Q3 = data.quantile([0.25, 0.75])
    IQR = Q3 - Q1
    iqr_lower = Q1 - 1.5 * IQR
    iqr_upper = Q3 + 1.5 * IQR

    z_scores = np.abs(stats.zscore(data))
    zscore_mask = z_scores > 3
    iqr_mask = (data < iqr_lower) | (data > iqr_upper)

    print(f'=== ANÁLISE DE OUTLIERS: {target} ===')
    print(f'Total de observações: {len(data):,}')
    print(f'IQR bounds: [{iqr_lower:.1f}, {iqr_upper:.1f}]')
    print(f'Outliers por IQR:    {iqr_mask.sum():,} ({iqr_mask.mean()*100:.1f}%)')
    print(f'Outliers por Z>3:    {zscore_mask.sum():,} ({zscore_mask.mean()*100:.1f}%)')

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f'🔍 Análise de Outliers — {target}', fontsize=15, fontweight='bold')

    # ① Boxplot com outliers destacados
    axes[0].boxplot(data, vert=True, patch_artist=True,
                    boxprops=dict(facecolor='#90E0EF'),
                    medianprops=dict(color='#FF6B6B', linewidth=2),
                    flierprops=dict(marker='o', markersize=3,
                                    markerfacecolor='#FF6B6B', alpha=0.3))
    axes[0].set_title('Boxplot com Outliers')

    # ② Histogram com limites
    axes[1].hist(data, bins=80, color='#90E0EF', edgecolor='white', alpha=0.8)
    axes[1].axvline(iqr_upper, color='#FF6B6B', ls='--', lw=2, label=f'IQR Upper ({iqr_upper:.1f})')
    axes[1].axvline(data.quantile(0.99), color='#FFA552', ls=':', lw=2, label='P99')
    axes[1].set_xlim(0, data.quantile(0.995))
    axes[1].set_title('Histograma com Limites')
    axes[1].legend()

    # ③ Percentis
    percentis = [50, 75, 90, 95, 97, 99, 99.5, 99.9]
    vals = [data.quantile(p / 100) for p in percentis]
    axes[2].plot(percentis, vals, 'o-', color=PALETTE[0], linewidth=2, markersize=8)
    axes[2].fill_between(percentis, vals, alpha=0.2, color=PALETTE[2])
    for p, v in zip(percentis, vals):
        axes[2].text(p, v, f'{v:.0f}', ha='center', va='bottom', fontsize=8)
    axes[2].set_xlabel('Percentil')
    axes[2].set_ylabel('Valor')
    axes[2].set_title('Curva de Percentis')
    axes[2].set_xticks(percentis)

    plt.tight_layout()
    plt.savefig('../reports/figures/11_outliers.png', bbox_inches='tight')
    plt.show()

## 10. Sugestões de Feature Engineering

In [ ]:
# ── Feature Engineering Sugerido ──────────────────────────────────────────
print("""
╔══════════════════════════════════════════════════════════════════════════╗
║         💡 SUGESTÕES DE FEATURE ENGINEERING PARA O TCC                 ║
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                          ║
║  TEMPORAIS:                                                              ║
║  • is_feriado: cruzar data com calendário de feriados nacionais/estaduais║
║  • semana_do_ano: ciclo anual                                            ║
║  • is_fim_mes / is_inicio_mes: pressão de fechamento de carga            ║
║  • hora_atracacao: turno (manhã/tarde/noite/madrugada)                   ║
║  • lag features: lead_time_anterior por porto                            ║
║                                                                          ║
║  CAPACIDADE / EMBARCAÇÃO:                                                ║
║  • log_dwt: log do porte bruto (normaliza skew)                          ║
║  • dwt_por_berco: carga relativa ao berço                                ║
║  • ratio_carga_capacidade: ocupação do navio                             ║
║                                                                          ║
║  PORTO / OPERAÇÃO:                                                       ║
║  • fila_porto: estimativa de congestionamento (n_navios simultâneos)     ║
║  • taxa_ocupacao_berco: berços ocupados / total berços no dia            ║
║  • lead_time_media_porto_mes: histórico do porto no mês                  ║
║                                                                          ║
║  GEOGRÁFICAS:                                                            ║
║  • distancia_ao_hub: km até Santos/Paranaguá/Suape                      ║
║  • is_porto_norte: portos fluviais têm dinâmica diferente                ║
║  • regiao_destino_carga: origem/destino previsto da carga                ║
║                                                                          ║
║  CLIMÁTICAS (fonte: INMET/Copernicus):                                   ║
║  • precipitacao_mm: chuva no dia/semana (atrasa operações)               ║
║  • velocidade_vento: afeta operações de granel sólido                    ║
║  • ondulacao_significativa: especialmente portos costeiros               ║
║  • nivel_rio: crítico para portos fluviais (Manaus, Belém, Itacoatiara)  ║
║  • estacao_climatica: codificada (verão/inverno/outono/primavera)        ║
║                                                                          ║
║  ECONÔMICAS:                                                             ║
║  • preco_soja / preco_milho / preco_minerio: commodities principais      ║
║  • cambio_dolar: afeta volumes de exportação                             ║
║  • indice_atividade_portuaria: proxy de demanda agregada                 ║
╚══════════════════════════════════════════════════════════════════════════╝
""")

In [ ]:
# ── Exemplo de Feature Engineering aplicado ──────────────────────────────
df_fe = df.copy()

# 1. Features temporais (se DATE_COL foi identificada)
if DATE_COL and DATE_COL in df_fe.columns:
    df_fe['turno_atracacao'] = pd.cut(df_fe['hora_dia'],
                                       bins=[0, 6, 12, 18, 24],
                                       labels=['Madrugada', 'Manhã', 'Tarde', 'Noite'],
                                       right=False)
    df_fe['is_fim_semana'] = df_fe['dow'].isin([5, 6]).astype(int)
    df_fe['is_fim_mes'] = (df_fe[DATE_COL].dt.day >= 25).astype(int)
    df_fe['semana_do_ano'] = df_fe[DATE_COL].dt.isocalendar().week.astype(int)

# 2. Log do porte bruto
if DWT_COL and DWT_COL in df_fe.columns:
    df_fe[f'log_{DWT_COL}'] = np.log1p(df_fe[DWT_COL])

# 3. Lead time histórico por porto (rolling lag feature)
if PORTO_COL and TARGET_COLS and DATE_COL:
    target = TARGET_COLS[0]
    df_fe = df_fe.sort_values(DATE_COL)
    df_fe['lt_media_porto_lag30d'] = (
        df_fe.groupby(PORTO_COL)[target]
             .transform(lambda x: x.shift(1).rolling(30, min_periods=5).mean())
    )

new_features = [c for c in df_fe.columns if c not in df.columns]
print(f'✅ {len(new_features)} novas features criadas:')
for f in new_features:
    print(f'   • {f}')

df_fe[new_features].describe()

## 11. Análise de Congestionamento Portuário

In [ ]:
# ── Estimativa de Congestionamento: Navios Simultâneos por Porto ──────────
# Assumindo colunas de início e fim de atracação
DT_INICIO = None
DT_FIM = None
for c_ini in ['dt_atracacao', 'dt_chegada', 'data_atracacao']:
    if c_ini in df.columns:
        DT_INICIO = c_ini
        break

for c_fim in ['dt_desatracacao', 'dt_saida', 'data_desatracacao', 'dt_fim_operacao']:
    if c_fim in df.columns:
        DT_FIM = c_fim
        break

if DT_INICIO and DT_FIM and PORTO_COL:
    print('Calculando congestionamento diário por porto...')
    df['dt_ini_dt'] = pd.to_datetime(df[DT_INICIO], errors='coerce')
    df['dt_fim_dt'] = pd.to_datetime(df[DT_FIM], errors='coerce')

    # Para cada dia, conta navios simultâneos (simplificado)
    dates = pd.date_range(df['dt_ini_dt'].min(), df['dt_fim_dt'].max(), freq='D')
    congestion = []
    for porto, gdf in df.dropna(subset=['dt_ini_dt', 'dt_fim_dt']).groupby(PORTO_COL):
        for d in dates[:365]:  # Limite para exemplo
            n = ((gdf['dt_ini_dt'] <= d) & (gdf['dt_fim_dt'] >= d)).sum()
            if n > 0:
                congestion.append({'porto': porto, 'data': d, 'navios_simultaneos': n})

    cong_df = pd.DataFrame(congestion)
    print(f'Registros de congestionamento: {len(cong_df):,}')
    cong_df.head()
else:
    print('Para calcular congestionamento, são necessárias colunas de início e fim de atracação.')
    print(f'DT_INICIO identificado: {DT_INICIO}')
    print(f'DT_FIM identificado: {DT_FIM}')

## 12. Dashboard Resumo Final (Plotly)

In [ ]:
# ── Dashboard Resumo com Subplots Plotly ──────────────────────────────────
if TARGET_COLS:
    target = TARGET_COLS[0]

    fig = make_subplots(
        rows=2, cols=3,
        subplot_titles=[
            '① Distribuição do Lead Time',
            '② Top Portos por Lead Time',
            '③ Lead Time por Mês',
            '④ Lead Time por Estação',
            '⑤ Correlações com Target',
            '⑥ Volume por Tipo de Carga'
        ],
        vertical_spacing=0.18,
        horizontal_spacing=0.1
    )

    data_clean = df[target].dropna()

    # ① Histograma
    fig.add_trace(go.Histogram(
        x=data_clean, nbinsx=60, name='Lead Time',
        marker_color='#0077B6', opacity=0.8
    ), row=1, col=1)

    # ② Top Portos
    if PORTO_COL:
        top10 = df.groupby(PORTO_COL)[target].median().sort_values(ascending=False).head(10)
        fig.add_trace(go.Bar(
            y=top10.index, x=top10.values, orientation='h',
            name='Lead Time Mediano', marker_color='#023E8A'
        ), row=1, col=2)

    # ③ Por mês
    if 'mes' in df.columns:
        mes_med = df.groupby('mes')[target].median()
        meses = ['Jan','Fev','Mar','Abr','Mai','Jun','Jul','Ago','Set','Out','Nov','Dez']
        fig.add_trace(go.Bar(
            x=meses, y=mes_med.reindex(range(1, 13)).values,
            name='Mediana Mensal',
            marker_color=['#FF6B6B' if v > mes_med.median() else '#90E0EF'
                          for v in mes_med.reindex(range(1, 13)).values]
        ), row=1, col=3)

    # ④ Por estação
    if 'estacao' in df.columns:
        est = df.groupby('estacao')[target].median().reindex(['Verão','Outono','Inverno','Primavera'])
        fig.add_trace(go.Bar(
            x=est.index.tolist(), y=est.values,
            marker_color=['#FF6B6B', '#FFA552', '#0077B6', '#06D6A0'],
            name='Lead Time por Estação'
        ), row=2, col=1)

    # ⑤ Correlações
    if len(num_cols) > 1:
        target_corr = df[num_cols].corr()[target].drop(target).abs().sort_values(ascending=False).head(10)
        fig.add_trace(go.Bar(
            x=target_corr.values, y=target_corr.index,
            orientation='h', marker_color='#00B4D8',
            name='|Correlação|'
        ), row=2, col=2)

    # ⑥ Volume por tipo de carga
    if CARGA_COL and CARGA_COL in df.columns:
        carga_vol = df[CARGA_COL].value_counts().head(10)
        fig.add_trace(go.Bar(
            x=carga_vol.index.tolist(), y=carga_vol.values,
            marker_color='#ADE8F4', name='Volume'
        ), row=2, col=3)

    fig.update_layout(
        height=800,
        title=dict(
            text='🚢 Dashboard EDA — Lead Time Portuário (TCC)',
            font=dict(size=20, color='#023E8A')
        ),
        showlegend=False,
        paper_bgcolor='white',
        plot_bgcolor='#f8f9fa'
    )

    fig.write_html('../reports/figures/dashboard_eda_completo.html')
    fig.show()
    print('✅ Dashboard salvo em ../reports/figures/dashboard_eda_completo.html')

---
## 📝 Próximos Passos Sugeridos

1. **Enriquecimento com dados externos:**
   - `INMET`: precipitação, temperatura, vento por coordenada/data → API pública
   - `ANTAQ`: dados oficiais de movimentação portuária (sistema SNOP)
   - `ANTT`: fluxo de cargas rodovias/ferrovias que alimentam os portos
   - `Banco Central`: séries cambiais (PTAX) e índices de atividade econômica (IBC-Br)
   - `IBGE`: IPCA, PIB regional, índice de atividade industrial por estado
   - `Copernicus Marine`: nível do mar, ondulação significativa
   - `AIS/MarineTraffic`: dados de chegada de navios (estimativas de chegada)

2. **Modelagem:**
   - Baseline: regressão linear + Ridge/Lasso
   - Ensemble: XGBoost, LightGBM (muito bons para dados tabulares)
   - Tratamento de target: testar `log(y)` vs `y` vs `sqrt(y)`
   - Validação temporal: split por data, não aleatório!
   - Métricas: MAE, RMSE, MAPE, e também quantílicas (P90, P95)

3. **Análise de impacto no capital de giro:**
   - Traduzir redução de lead time em dias de estoque → R$ de capital liberado
   - Sensibilidade por tipo de carga/porto/cliente

4. **Estrutura recomendada de notebooks:**
   - `01_ingestion.ipynb` (já tem via script)
   - `02_EDA.ipynb` ← este notebook
   - `03_feature_engineering.ipynb`
   - `04_modeling_baseline.ipynb`
   - `05_modeling_advanced.ipynb`
   - `06_interpretabilidade_shap.ipynb`
   - `07_resultados_negocio.ipynb`

---
*Notebook gerado para o TCC de Predição de Lead Time Portuário*